In [115]:
import sqlite3
import pandas as pd

In [116]:
conn = sqlite3.connect("download.db")
Qu1_sql = """SELECT m.member_id, m.first_name, m.last_name, COUNT(c.checkout_id) as total_checkouts
FROM members m
LEFT JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id;
"""
reslt_Qu1 = pd.read_sql_query(Qu1_sql, conn)
print("\nQ1 Result:\n", reslt_Qu1)



Q1 Result:
     member_id first_name last_name  total_checkouts
0        1001      Salma   Ibrahim                1
1        1002      Fares     Saleh                2
2        1003     Bassel    Hegazy                9
3        1004      Fares     Wahba                0
4        1005    Youssef     Halim                3
..        ...        ...       ...              ...
75       1076       Dina     Wahba                7
76       1077       Lina    Rashad                6
77       1078     Habiba     Osman                0
78       1079       Rana     Osman               10
79       1080     Bassel     Wahba                2

[80 rows x 4 columns]


In [117]:
Qu2_sql= "SELECT * FROM books WHERE author LIKE '%Aya%';"
reslt_Qu2 = pd.read_sql_query(Qu2_sql, conn)
print("\nQ2 Result:\n", reslt_Qu2)


Q2 Result:
    book_id                title     author
0      505  Letters to the Nile  Aya Hafez
1      506  The Paper Boat Club  Aya Hafez


In [118]:
QU3_sql = """
SELECT b.title, COUNT(c.checkout_id) as checkouts
FROM checkouts c
JOIN books b ON c.book_id = b.book_id
GROUP BY b.book_id
ORDER BY checkouts DESC
LIMIT 5;
"""
reslt_Qu3 = pd.read_sql_query(QU3_sql, conn)
print("\nQ3 Result:\n", reslt_Qu3)


Q3 Result:
                     title  checkouts
0         The Silver Kite         57
1   Fossils and Fireflies         55
2  Circuits for Beginners         46
3        Kites Over Cairo         38
4    Storms and Sailboats         25


In [119]:
Qu4_sql = """
SELECT m.first_name, m.last_name, 
COUNT(c.checkout_id) as checkouts
FROM checkouts c 
JOIN members m ON c.member_id = m.member_id
GROUP BY m.member_id 
ORDER BY checkouts DESC 
LIMIT 10;
"""
result_QU4 = pd.read_sql_query(Qu4_sql, conn)
print("\nQ4 Result:\n", result_QU4)


Q4 Result:
   first_name last_name  checkouts
0        Aya     Wahba         25
1     Sherif     Saleh         21
2       Ziad     Saleh         19
3    Mostafa     Fouad         18
4       Nour     Nabil         18
5       Adam     Fahmy         17
6    Youssef    Hegazy         17
7      Ahmed    Shafik         17
8       Sara    Rashad         16
9       Reem     Osman         16


In [120]:
Q5_sql = """
SELECT c.*, m.neighborhood
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
WHERE m.neighborhood = 'Maadi'
ORDER BY c.checkout_date DESC;
"""

res_q5 = pd.read_sql_query(Q5_sql, conn)
rest_Q5 = res_q5.iloc[10:]
print("\nQ5 Result:\n", rest_Q5.head())




Q5 Result:
     checkout_id  member_id  book_id checkout_date return_date neighborhood
10         9103       1003      513    2025-09-04  2025-09-25        Maadi
11         9081       1017      502    2025-08-25  2025-09-17        Maadi
12         9001       1008      501    2025-08-23  2025-08-28        Maadi
13         9050       1003      521    2025-08-21         NaN        Maadi
14         9085       1018      525    2025-08-19  2025-09-18        Maadi


In [121]:
Db_members = pd.read_sql_query("SELECT * FROM members", conn)
Db_books = pd.read_sql_query("SELECT * FROM books", conn)
Db_checkouts = pd.read_sql_query("SELECT * FROM checkouts", conn)
Json_books = pd.read_json("download.json")
Html_kickoff = pd.read_html("download.html")[0]
Html_kickoff.columns = ["member_id", "book_id", "checkout_date"]
Html_kickoff["return_date"] = None
allbooks = pd.merge(Db_books, Json_books, how="left", on="book_id")
merged_db = Db_checkouts.merge(Db_members, on="member_id", how="left").merge(allbooks, on="book_id", how="left")
merged_Html = Html_kickoff.merge(Db_members, on="member_id", how="left").merge(allbooks, on="book_id", how="left")
final_df = pd.concat([merged_db, merged_Html], ignore_index=True)
checkout_counts = final_df["member_id"].value_counts()
final_df["member_total_checkouts"] = final_df["member_id"].map(checkout_counts)
final_df.to_csv("task1_combined_data.csv", index=False)

In [122]:
dff= pd.read_csv("task1_combined_data.csv")
print(dff.head())


   checkout_id  member_id  book_id checkout_date return_date first_name  \
0       9263.0       1047      517    2024-10-21  2024-11-07       Sara   
1       9340.0       1072      513    2025-08-24  2025-09-01       Seif   
2       9231.0       1053      523    2024-02-04  2024-02-16       Adam   
3       9129.0       1032      513    2025-06-21  2025-06-29       Nada   
4       9370.0       1079      511    2025-11-11  2025-12-03       Rana   

  last_name  grade neighborhood membership_status   join_date  \
0    Rashad    NaN   Heliopolis          Inactive  2024-06-25   
1      Zaki    9.0      Zamalek            Active  2025-10-21   
2    Shafik    9.0   Heliopolis            Active  2024-01-03   
3      Zaki    7.0    Nasr City            Active  2025-10-19   
4     Osman    8.0       Shubra            Active  2024-10-27   

                     title        author       genre  pages  publication_year  \
0  Shadows on the Corniche   Hani Nagati     Mystery    338            2015.0

In [123]:

print(dff.describe())


       checkout_id    member_id     book_id       grade       pages  \
count   391.000000   417.000000  417.000000  376.000000  417.000000   
mean   9192.611253  1041.004796  513.443645    7.577128  202.299760   
std     110.279771    26.356925    8.976053    1.154427   73.986738   
min    9001.000000  1001.000000  501.000000    6.000000  104.000000   
25%    9097.500000  1020.000000  507.000000    7.000000  134.000000   
50%    9193.000000  1036.000000  513.000000    8.000000  160.000000   
75%    9287.500000  1061.000000  519.000000    9.000000  294.000000   
max    9383.000000  1201.000000  532.000000    9.000000  338.000000   

       publication_year  member_total_checkouts  
count        382.000000              417.000000  
mean        2017.426702               11.968825  
std            4.567214                6.972231  
min         2009.000000                1.000000  
25%         2014.000000                6.000000  
50%         2017.000000               11.000000  
75%       

In [124]:

print(dff.info())
df = dff.copy()
df['return_date'] = df['return_date'].fillna("Not Returned")
df['join_date'] = df['join_date'].fillna("Unknown")
df = df.dropna(subset=['first_name'])
max_checkout_id = df['checkout_id'].max()
for id in df[df['checkout_id'].isnull()].index:
    max_checkout_id += 1
    df.loc[id, 'checkout_id'] = max_checkout_id
df['checkout_id'] = df['checkout_id'].astype(int)
print(df.info())


<class 'pandas.DataFrame'>
RangeIndex: 417 entries, 0 to 416
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   checkout_id             391 non-null    float64
 1   member_id               417 non-null    int64  
 2   book_id                 417 non-null    int64  
 3   checkout_date           417 non-null    str    
 4   return_date             326 non-null    str    
 5   first_name              412 non-null    str    
 6   last_name               412 non-null    str    
 7   grade                   376 non-null    float64
 8   neighborhood            412 non-null    str    
 9   membership_status       412 non-null    str    
 10  join_date               406 non-null    str    
 11  title                   417 non-null    str    
 12  author                  417 non-null    str    
 13  genre                   417 non-null    str    
 14  pages                   417 non-null    int64  
 15  

In [131]:
df_duplicated = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Number of duplicated rows: {df_duplicated}")

Number of duplicated rows: 0


In [132]:
df["neighborhood"] = df["neighborhood"].astype(str).str.strip().str.lower().str.title()
print(df["neighborhood"].value_counts())
df["membership_status"] = df["membership_status"].astype(str).str.strip().str.title()
print(df["membership_status"].value_counts())

neighborhood
Maadi         114
Nasr City     101
Heliopolis     87
Zamalek        68
Shubra         34
Name: count, dtype: int64
membership_status
Active      320
Inactive     84
Name: count, dtype: int64


In [133]:
df.info()

<class 'pandas.DataFrame'>
Index: 404 entries, 0 to 416
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   checkout_id             404 non-null    int64  
 1   member_id               404 non-null    int64  
 2   book_id                 404 non-null    int64  
 3   checkout_date           404 non-null    str    
 4   return_date             404 non-null    str    
 5   first_name              404 non-null    str    
 6   last_name               404 non-null    str    
 7   grade                   368 non-null    float64
 8   neighborhood            404 non-null    str    
 9   membership_status       404 non-null    str    
 10  join_date               404 non-null    str    
 11  title                   404 non-null    str    
 12  author                  404 non-null    str    
 13  genre                   404 non-null    str    
 14  pages                   404 non-null    int64  
 15  publi

In [134]:
df.to_csv("task2_cleaned_data.csv", index=False)